# Modulating QMzymeRegion
## Objective

The objective of this tutorial is to show different ways in which QMzymeRegion can be modified. We will highlight some of the ways in which the user can manipulate QMzymeRegion to get a desirable selection for QM input generation. This workflow allows you to:

- Learn methods to combine and subtract QMzymeRegion objects.
- Learn how to add acetyl (ACE) and N-methyl amide (NME) cap to the residue.

In this specific example, we are using ketosteroid isomerase (KSI) as the model system. The structure for KSI is obtained from the PDB [1OH0](https://doi.org/10.2210/pdb1OH0/pdb) and MM-minimized prior to this tutorial.

## Classes used in this example

- [GenerateModel](https://qmzyme.readthedocs.io/en/latest/API/QMzyme.GenerateModel.html)
- [SelectionSchemes](https://qmzyme.readthedocs.io/en/latest/API/QMzyme.SelectionSchemes.html)
    - [DistanceCutoff SelectionScheme](https://qmzyme.readthedocs.io/en/latest/API/QMzyme.SelectionSchemes.html#QMzyme.SelectionSchemes.DistanceCutoff)
- [QMzymeRegion](https://qmzyme.readthedocs.io/en/latest/API/QMzyme.QMzymeRegion.html)
    - [combine method](https://qmzyme.readthedocs.io/en/latest/API/QMzyme.QMzymeRegion.html#QMzyme.QMzymeRegion.QMzymeRegion.combine)
    - [subtract method](https://qmzyme.readthedocs.io/en/latest/API/QMzyme.QMzymeRegion.html#QMzyme.QMzymeRegion.QMzymeRegion.subtract)
- [TruncationSchemes](https://qmzyme.readthedocs.io/en/latest/API/QMzyme.TruncationSchemes.html#QMzyme.TruncationSchemes.TruncationScheme)
    - [BetaCarbon TruncationScheme](https://qmzyme.readthedocs.io/en/latest/API/QMzyme.TruncationSchemes.html#QMzyme.TruncationSchemes.BetaCarbon)

## Required Files
To start, you will need:

- A fully prepped and protonated PDB
  
---

In [ ]:
# Here are the necesary imports for this tutorial!

import QMzyme
from QMzyme import GenerateModel
from QMzyme.SelectionSchemes import DistanceCutoff
from QMzyme.data import PDB
from QMzyme.RegionBuilder import RegionBuilder
import pandas as pd
import MDAnalysis

## Combining Two Regions

We'll first look at combining two regions! To achieve this, we can use `combine()` method in QMzymeRegion class. Using it is quite simple: you decide on the base region and region you want to add, then simply combine them using `combine()`. In here, we will use it to add Tyr 57 to our distance cutoff of 3 Å.

In [2]:
# We first initialize model and update the unknown residue charge.
model = QMzyme.GenerateModel(PDB)
QMzyme.data.residue_charges.update({'EQU': -1}) 

# We create regions of interest.
model.set_catalytic_center(selection='resname EQU and segid A')
model.set_region(selection=DistanceCutoff, cutoff=2)
model.set_region(selection="resid 57", name="Tyr_57")

# We combine Tyr_57 region to cutoff_3 region.
combined_region = model.get_region("cutoff_2")
combined_region = combined_region.combine(model.Tyr_57)
model.set_region(selection=combined_region, name=f"combined_region")


Charge information not present. QMzyme will try to guess region charges based on residue names consistent with AMBER naming conventions (i.e., aspartate: ASP --> Charge: -1, aspartic acid: ASH --> Charge: 0.). See QMzyme.data.residue_charges for the full set.

	Nonconventional Residues Found
	------------------------------
	EQU --> Charge: UNK, defaulting to 0

You can update charge information for nonconventional residues by running 
	>>>QMzyme.data.residue_charges.update({'3LETTER_RESNAME':INTEGER_CHARGE}). 
Note your changes will not be stored after you exit your session. It is recommended to only alter the residue_charges dictionary. If you alter the protein_residues dictionary instead that could cause unintended bugs in other modules (TruncationSchemes).



We can examine the region using pandas and `summarize()` method. We'll first look at our cutoff_3 region, then compare it with combined_region!

In [3]:
df = pd.DataFrame(model.cutoff_2.summarize())
df

,Resid,Resname,Charge,Removed atoms,Added atoms,Fixed atoms,Truncation scheme,Capping scheme,Method,Segids
0,16,TYR,0,[],[],[],None,None,None,A
1,103,ASH,0,[],[],[],None,None,None,A
2,263,EQU,-1,[],[],[],None,None,None,A
3,373,WAT,0,[],[],[],None,None,None,A


In [4]:
df = pd.DataFrame(model.combined_region.summarize())
df

,Resid,Resname,Charge,Removed atoms,Added atoms,Fixed atoms,Truncation scheme,Capping scheme,Method,Segids
0,16,TYR,0,[],[],[],None,None,None,A
1,57,TYR,0,[],[],[],None,None,None,A
2,103,ASH,0,[],[],[],None,None,None,A
3,263,EQU,-1,[],[],[],None,None,None,A
4,373,WAT,0,[],[],[],None,None,None,A


In [5]:
model.print_overview()

-----------------------------
Model Overview: 1oh0 
-----------------------------
  - total atoms: 4258
  - total residues: 324
  - total regions: 4
-----------------------------
Region Overview
-----------------------------
Region Name: catalytic_center
  - atoms: 37
  - residues: 1
  - method: None
  - selection_scheme: resname EQU and segid A
-----------------------------
Region Name: cutoff_2
  - atoms: 74
  - residues: 4
  - method: None
  - selection_scheme: DistanceCutoff
  - cutoff: 2
-----------------------------
Region Name: Tyr_57
  - atoms: 21
  - residues: 1
  - method: None
  - selection_scheme: resid 57
-----------------------------
Region Name: combined_region
  - atoms: 95
  - residues: 5
  - method: None
  - selection_scheme: cutoff_2 + Tyr_57
-----------------------------


As you can see, Tyr 57 can be seen in combined_region, suggesting that our region has been successfully combined!

## Subtracting Two Regions

Now, let's subtract a region from our QMzyme region! To achieve this, we can use `subtract()` method in QMzymeRegion class. This time, we'll consider a case where you want to remove amino acid residues responsible for creating the oxyanion hole in KSI (Tyr 16 and Asp 103) to see how it influences coordination of the substrate.

In [6]:
# We first initialize model and update the unknown residue charge.
model = QMzyme.GenerateModel(PDB)
QMzyme.data.residue_charges.update({'EQU': -1}) 

# We create regions of interest.
model.set_catalytic_center(selection='resname EQU and segid A')
model.set_region(selection=DistanceCutoff, cutoff=2)
model.set_region(selection="resid 16 or resid 103", name="oxyanion_hole")

# We combine Tyr_57 region to cutoff_3 region.
subtracted_region = model.get_region("cutoff_2")
subtracted_region = subtracted_region.subtract(model.oxyanion_hole)
model.set_region(selection=subtracted_region, name=f"subtracted_region")


Charge information not present. QMzyme will try to guess region charges based on residue names consistent with AMBER naming conventions (i.e., aspartate: ASP --> Charge: -1, aspartic acid: ASH --> Charge: 0.). See QMzyme.data.residue_charges for the full set.


We can examine the region using pandas and `summarize()` method. We'll first look at our cutoff_3 region, then compare it with subtracted_region!

In [7]:
df = pd.DataFrame(model.cutoff_2.summarize())
df

,Resid,Resname,Charge,Removed atoms,Added atoms,Fixed atoms,Truncation scheme,Capping scheme,Method,Segids
0,16,TYR,0,[],[],[],None,None,None,A
1,103,ASH,0,[],[],[],None,None,None,A
2,263,EQU,-1,[],[],[],None,None,None,A
3,373,WAT,0,[],[],[],None,None,None,A


In [8]:
df = pd.DataFrame(model.subtracted_region.summarize())
df

,Resid,Resname,Charge,Removed atoms,Added atoms,Fixed atoms,Truncation scheme,Capping scheme,Method,Segids
0,263,EQU,-1,[],[],[],None,None,None,A
1,373,WAT,0,[],[],[],None,None,None,A


In [9]:
model.print_overview()

-----------------------------
Model Overview: 1oh0 
-----------------------------
  - total atoms: 4258
  - total residues: 324
  - total regions: 4
-----------------------------
Region Overview
-----------------------------
Region Name: catalytic_center
  - atoms: 37
  - residues: 1
  - method: None
  - selection_scheme: resname EQU and segid A
-----------------------------
Region Name: cutoff_2
  - atoms: 74
  - residues: 4
  - method: None
  - selection_scheme: DistanceCutoff
  - cutoff: 2
-----------------------------
Region Name: oxyanion_hole
  - atoms: 34
  - residues: 2
  - method: None
  - selection_scheme: resid 16 or resid 103
-----------------------------
Region Name: subtracted_region
  - atoms: 40
  - residues: 2
  - method: None
  - selection_scheme: cutoff_2 - oxyanion_hole
-----------------------------


As you can see, Tyr 16 and Asp 103 are no longer present in subtracted_region, suggesting that our region has been successfully subtracted!

## Adding ACE and NME capping

In cases where the user wants to add acetal (ACE) or N-methyl amide (NME) capping, the user can use `add_N_terminus_ACE()` or `add_C_terminus_NME()` to add the apporpriate capping.

In [10]:
import QMzyme
from QMzyme.data import PDB
import pandas as pd

model = QMzyme.GenerateModel(PDB)
QMzyme.data.residue_charges.update({'EQU': -1}) 
model.set_region(selection="resid 4 or resid 6", name="test")


Charge information not present. QMzyme will try to guess region charges based on residue names consistent with AMBER naming conventions (i.e., aspartate: ASP --> Charge: -1, aspartic acid: ASH --> Charge: 0.). See QMzyme.data.residue_charges for the full set.


In [12]:
df = pd.DataFrame(model.test.summarize())
df

,Resid,Resname,Charge,Removed atoms,Added atoms,Fixed atoms,Truncation scheme,Capping scheme,Method,Segids
0,4,PRO,0,[],[],[],None,None,None,A
1,6,ALA,0,[],[],[],None,None,None,A


To add ACE capping to a residue, use `add_N_terminus_ACE()`.

In [13]:
model.test.add_N_terminus_ACE(resid = 4)

In [14]:
df = pd.DataFrame(model.test.summarize())
df

,Resid,Resname,Charge,Removed atoms,Added atoms,Fixed atoms,Truncation scheme,Capping scheme,Method,Segids
0,3,ACE,0,"[N, H, CA, HA, CB, HB2, HB3, CG, HG, CD1, HD11...","[HH32, CH3, HH31, HH33]",[],None,None,None,A
1,4,PRO,0,[],[],[],None,cap_ACE,None,A
2,6,ALA,0,[],[],[],None,None,None,A


To add NME capping to a residue, use `add_C_terminus_NME()`.

In [15]:
model.test.add_C_terminus_NME(resid = 6)

In [16]:
df = pd.DataFrame(model.test.summarize())
df

,Resid,Resname,Charge,Removed atoms,Added atoms,Fixed atoms,Truncation scheme,Capping scheme,Method,Segids
0,3,ACE,0,"[N, H, CA, HA, CB, HB2, HB3, CG, HG, CD1, HD11...","[HH32, CH3, HH31, HH33]",[],None,None,None,A
1,4,PRO,0,[],[],[],None,cap_ACE,None,A
2,6,ALA,0,[],[],[],None,cap_NME,None,A
3,7,NME,0,"[CA, HA, CB, HB2, HB3, CG, HG2, HG3, CD, OE1, ...","[CH3, HH31, HH33, HH32]",[],None,None,None,A


If both ACE and NME capping are applied to the region, the backbone of the residue will be preserved. This is done by applying `BetaCarbon()` TruncationScheme subclass, which will mutate the residue to alanine.

⚠ **Important considerations:**
- If the in-between residue is glycine or proline, the whole residue will be included.

In [17]:
model.test.add_C_terminus_NME(resid = 4)
model.test.add_N_terminus_ACE(resid = 6)

In [18]:
df = pd.DataFrame(model.test.summarize())
df

,Resid,Resname,Charge,Removed atoms,Added atoms,Fixed atoms,Truncation scheme,Capping scheme,Method,Segids
0,3,ACE,0,"[N, H, CA, HA, CB, HB2, HB3, CG, HG, CD1, HD11...","[HH32, CH3, HH31, HH33]",[],None,None,None,A
1,4,PRO,0,[],[],[],None,cap_ACE,None,A
2,5,ALA,0,"[HB, CG2, HG21, HG22, HG23, OG1]","[HB1, HB2, HB3]",[],BetaCarbon,None,None,A
3,6,ALA,0,[],[],[],None,cap_NME,None,A
4,7,NME,0,"[CA, HA, CB, HB2, HB3, CG, HG2, HG3, CD, OE1, ...","[CH3, HH31, HH33, HH32]",[],None,None,None,A
